## Trabajo Práctico Grupal · Cámara de Diputados, 2022–2026


In [ ]:
import collections
import math
import re
import unicodedata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.special import log_ndtr
import igraph as ig
from wordcloud import WordCloud

## Colores por bloque

Cada bloque tiene un color fijo en todo el trabajo, elegido según el color con
el que se lo identifica en la política argentina: violeta para La Libertad
Avanza, amarillo para el PRO, celeste para el peronismo (Frente de Todos y su
sucesor, Unión por la Patria), rojo para la UCR, bordó para el Frente de
Izquierda y naranja para Provincias Unidas. Los bloques sin un color
establecido (desprendimientos de la UCR, bloques provinciales) van en tonos que
no compitan con esos: rosados los que salen del radicalismo, neutros el resto.

El celeste del peronismo y el rojo de la UCR son más claro y más oscuro,
respectivamente, que el azul y rojo de la escala izquierda/derecha del mapa de
proyectos (`IZQUIERDA` en rojo, `DERECHA` en azul), para que no se confundan
con esa polaridad.

In [ ]:
NOMBRE_BLOQUE = {
    "Frente De Todos": "Frente de Todos",
    "Union Por La Patria": "Unión por la Patria",
    "La Libertad Avanza": "La Libertad Avanza",
    "Pro": "PRO",
    "Ucr": "UCR",
    "Ucr - Union Civica Radical": "UCR",
    "Evolucion Radical": "Evolución Radical",
    "Coalicion Civica": "Coalición Cívica",
    "Encuentro Federal": "Encuentro Federal",
    "Democracia Para Siempre": "Democracia para Siempre",
    "Provincias Unidas": "Provincias Unidas",
    "Innovacion Federal": "Innovación Federal",
}

COLOR_BLOQUE = {
    "La Libertad Avanza": "#7b3fb5",
    "PRO": "#f0b800",
    "Frente de Todos": "#5bb8ec",
    "Unión por la Patria": "#5bb8ec",
    "UCR": "#c8102e",
    "Frente de Izquierda": "#7a1f2b",
    "Provincias Unidas": "#f07c1e",
    "Coalición Cívica": "#1baf7a",
    "Evolución Radical": "#e87ba4",
    "Democracia para Siempre": "#c45a8c",
    "Encuentro Federal": "#6f8296",
    "Innovación Federal": "#a3845c",
    "Otros bloques": "#c3c2b7",
}

## Cargar los puntos ideales


In [ ]:
df = pd.read_csv("datos/idps.csv")
df.head()

## Orientar el PCA

El signo de `idp_pca` es arbitrario por año. Lo reorientamos acá
contra `idp_bayes`, que sí viene fijado contra legisladores de referencia de
izquierda/derecha: si la correlación
entre los dos métodos da negativa para un año, se invierte el signo de
`idp_pca` ese año.

In [ ]:
def orientar_pca(df):
    df = df.copy()
    for anio, g in df.groupby("anio"):
        r = g["idp_pca"].corr(g["idp_bayes"])
        if r is not None and r < 0:
            df.loc[g.index, "idp_pca"] = -df.loc[g.index, "idp_pca"]
    return df


df = orientar_pca(df)

## Bloque de cada diputado

El bloque de cada legislador ese año sale de `legisladores_bloque_provincia.csv`.
Todos los bloques del Frente de Izquierda (PTS, Partido Obrero, Izquierda
Socialista, MST: 1 o 2 bancas cada uno, con nombres que cambian de un año a
otro) se agrupan bajo "Frente de Izquierda" —así se los suele nombrar en
conjunto— para que el color tenga sentido como categoría. El resto de los
bloques con menos de 10 legisladores *ese año* va a "Otros bloques", igual que
los pocos legisladores que no aparecen en ese archivo. Los colores son los de
`COLOR_BLOQUE` (definidos al principio), así que cada bloque se ve igual en
todas las solapas y en todos los gráficos.

In [ ]:
bloques = pd.read_csv("datos/legisladores_bloque_provincia.csv")

df = df.merge(bloques[["legislador", "yr", "bloque"]], left_on=["nombre", "anio"], right_on=["legislador", "yr"],
              how="left").drop(columns=["legislador", "yr"])
df["bloque"] = df["bloque"].where(~df["bloque"].str.contains("izquierda", case=False, na=False), "Frente de Izquierda")
df["bloque"] = df["bloque"].replace(NOMBRE_BLOQUE)
grandes = df.groupby(["anio", "bloque"])["nombre"].transform("size") >= 10
df["bloque_mostrado"] = df["bloque"].where(grandes | (df["bloque"] == "Frente de Izquierda"), "Otros bloques")
assert df["bloque_mostrado"].isin(COLOR_BLOQUE).all(), set(df["bloque_mostrado"]) - set(COLOR_BLOQUE)
df.groupby("anio")["bloque_mostrado"].value_counts().unstack(0).fillna(0).astype(int)

## Puntos ideales por año: Bayes y PCA

In [ ]:
IZQUIERDA, DERECHA = "#e34948", "#2a78d6"
WHISKER, BASELINE, GRID = "#898781", "#c3c2b7", "#e1e0d9"
SURFACE, PAGE = "#fcfcfb", "#f9f9f7"
TEXT, TEXT2, MUTED = "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, \'Segoe UI\', sans-serif"


def panel(fig, row, col, g, columna, sd=None):
    """Un panel: barras divergentes desde 0, ordenadas por valor y coloreadas
    por bloque, con opcionalmente un whisker +-sd (Bayes) y hover de nombre,
    bloque e idp."""
    g = g.sort_values(columna)
    bloque = g["bloque"].fillna("sin bloque")
    kw = dict(
        x=g[columna], y=list(range(len(g))), orientation="h",
        marker=dict(color=g["bloque_mostrado"].map(COLOR_BLOQUE), line=dict(width=0)),
        showlegend=False,
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    )
    if sd is not None:
        kw["error_x"] = dict(type="data", array=g[sd], color=WHISKER, thickness=1, width=0)
        kw["customdata"] = list(zip(g["nombre"], bloque, g[sd]))
        kw["hovertemplate"] = "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{x:.2f} (\u00b1%{customdata[2]:.2f})<extra></extra>"
    else:
        kw["customdata"] = list(zip(g["nombre"], bloque))
        kw["hovertemplate"] = "<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{x:.2f}<extra></extra>"
    fig.add_trace(go.Bar(**kw), row=row, col=col)

In [ ]:
anios = sorted(df["anio"].unique())

bayes_domain = math.ceil((df["idp_bayes"].abs() + df["sd_bayes"]).max() * 10) / 10
pca_domain = 1.08

fig = make_subplots(
    rows=1, cols=2,
    column_titles=["Bayes · media posterior", "PCA · reescalado a [-1, 1], orientado contra Bayes"],
    horizontal_spacing=0.05,
)

trazas_fig = {}
for anio in anios:
    g = df[df["anio"] == anio]
    trazas_fig[anio] = [len(fig.data), len(fig.data) + 1]
    panel(fig, 1, 1, g, "idp_bayes", sd="sd_bayes")
    panel(fig, 1, 2, g.dropna(subset=["idp_pca"]), "idp_pca")
    # leyenda del año: trazas invisibles, una por bloque
    for bloque, gb in g.groupby("bloque_mostrado"):
        trazas_fig[anio].append(len(fig.data))
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
                                  marker=dict(size=10, color=COLOR_BLOQUE[bloque]),
                                  name=f"{bloque} ({len(gb)})",
                                  legendrank=1000 if bloque == "Otros bloques" else gb["idp_bayes"].median() * 100 + 500),
                      row=1, col=1)

fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                          line=dict(color=WHISKER, width=2),
                          name="\u00b1 1 desvio (incertidumbre, solo Bayes)", legendrank=1001), row=1, col=1)


def visibles(anio):
    return [i in trazas_fig[anio] for i in range(len(fig.data) - 1)] + [True]


inicial = anios[-1]
for traza, visible in zip(fig.data, visibles(inicial)):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=[{"visible": visibles(a)}]) for a in anios]

fig.update_xaxes(range=[-bayes_domain, bayes_domain], zeroline=True,
                  zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                  tickfont=dict(size=10, color=MUTED), row=1, col=1)
fig.update_xaxes(range=[-pca_domain, pca_domain], zeroline=True,
                  zerolinecolor=BASELINE, zerolinewidth=1, gridcolor=GRID,
                  tickfont=dict(size=10, color=MUTED), row=1, col=2)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, autorange="reversed")

fig.update_annotations(font=dict(size=13, color=TEXT2))

fig.update_layout(
    title=dict(text="Puntos ideales \u00b7 C\u00e1mara de Diputados, 2022\u20132026", x=0.01, y=0.98, yanchor="top",
               font=dict(size=20, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=900, width=1000,
    margin=dict(l=30, r=20, t=150, b=30),
    legend=dict(x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11)),
    bargap=0.1,
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.08, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig.show(config={"displaylogo": False, "scrollZoom": False})

**Desvíos grandes que no son ideología: Massa (2022) y Marcela Herrera (2023).**
En 2022, Sergio Massa tiene el desvío más grande de la Cámara (`sd_bayes` ≈ 0.80)
con un `idp_bayes` cerca de 0. No es que sea ambiguo: fue presidente de la Cámara
hasta agosto, y el presidente no vota, así que en sus registros solo figura
`PRESIDENTE` o `AUSENTE`, sin un solo Sí o No. Marcela Herrera, en 2023, aparece
en una sola votación (la última del año) y es unánime, y las unánimes quedan
afuera del modelo. En los dos casos el modelo no tiene ningún voto útil, y lo que
muestra es el prior: un punto cerca del centro con una incertidumbre del ancho
de todo el eje. Esa es una ventaja de Bayes sobre PCA: el PCA igual le asigna a
Herrera un valor (−0.50) que parece tan firme como cualquier otro, porque sale de
imputar los votos que le faltan. Bayes, en cambio, avisa que no sabe.

**El Frente de Izquierda en el centro (2022): el eje mide cercanía al gobierno.**
En 2022, Bregman, Del Caño y Del Pla quedan en `idp_bayes` ≈ +0.37, apenas a la
derecha del cero (puestos 137–140 de 259), mientras que Máximo Kirchner está en
−1.11 y Milei en +2.37. No es que el FIT esté "cerca de Milei": Bregman vota igual
que Kirchner en el 74% de las votaciones que comparten y que Milei en el 46%. Pero
también coincide un 74% con Negri (UCR), y eso la deja en el medio. El FIT
acompaña al Frente de Todos en muchas leyes sociales, pero vota en contra del
gobierno en las económicas centrales: Presupuesto 2023, Consenso Fiscal,
prórroga de impuestos, promoción automotriz, biotecnología. En esas votaciones
Bregman y Milei votan No y Kirchner Sí; el FIT se opone por izquierda y Milei por
derecha, pero el voto es el mismo. Un modelo de una sola dimensión no tiene cómo
representar "opositor por izquierda", y un diputado que a veces vota con el
oficialismo y a veces con la oposición termina en el promedio. Es una limitación
conocida (*ends against the middle*: los dos extremos votan juntos contra el
centro, por motivos opuestos), y muestra que en estos años el eje mide sobre todo
la distancia al oficialismo de turno. Por eso el FIT se corre a medida que cambia
el gobierno: en 2023 queda casi en el centro (−0.14), en 2024 y 2025 ya está del
lado izquierdo junto a Unión por la Patria, y en 2026 sus tres diputados son los
tres más a la izquierda de la Cámara. Con Milei en el gobierno, oponerse al
oficialismo y ser de izquierda pasan a ser lo mismo.

## Tabla de posiciones: los 10 desvíos más grandes de cada año

Los diez diputados con `sd_bayes` más alto en cada año, sin los casos como
Massa o Herrera: se dejan afuera los que tienen menos de 10 **votos útiles**
ese año, es decir, votos Sí o No en votaciones que no fueron unánimes (las
únicas que usa el modelo; ver `preparar_pscl()` en `03_puntos_ideales.py`).
El color de cada celda es el de su bloque.

In [ ]:
MIN_VOTOS_UTILES = 10

ses = pd.read_csv("datos/sesiones_dt.csv.gz", dtype={"legislador": str}, keep_default_na=False)

si_no = ses[ses["voto"].isin(["AFIRMATIVO", "NEGATIVO"])]
divididas = si_no.groupby("id_votacion")["voto"].nunique().loc[lambda n: n == 2].index
utiles = si_no[si_no["id_votacion"].isin(divididas)].groupby(["legislador", "yr"]).size()
df["votos_utiles"] = utiles.reindex(pd.MultiIndex.from_frame(df[["nombre", "anio"]])).fillna(0).astype(int).to_numpy()

top_desvio = (df[df["votos_utiles"] >= MIN_VOTOS_UTILES]
              .sort_values("sd_bayes", ascending=False)
              .groupby("anio").head(10))
excluidos = df[df["votos_utiles"] < MIN_VOTOS_UTILES].sort_values(["anio", "votos_utiles"])[
    ["anio", "nombre", "votos_utiles", "idp_bayes", "sd_bayes"]]
print(f"Fuera de la tabla de posiciones (menos de {MIN_VOTOS_UTILES} votos útiles): {len(excluidos)} diputados")
excluidos

In [ ]:
def suave(color, alfa=0.25):
    r, g, b = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r}, {g}, {b}, {alfa})"


textos, fondos = [list(range(1, 11))], [[SURFACE] * 10]
for anio, g in top_desvio.groupby("anio"):
    textos.append([f"<b>{n.title()}</b><br>{b} · sd {sd:.2f} · idp {idp:+.2f}"
                   for n, b, sd, idp in zip(g["nombre"], g["bloque"].fillna("sin bloque"), g["sd_bayes"], g["idp_bayes"])])
    fondos.append([suave(COLOR_BLOQUE[b]) for b in g["bloque_mostrado"]])

fig_desvio = go.Figure(go.Table(
    columnwidth=[0.25] + [1] * len(anios),
    header=dict(values=["#"] + [f"<b>{a}</b>" for a in anios], fill_color=PAGE, line_color=GRID,
                font=dict(size=13, color=TEXT), align="left", height=32),
    cells=dict(values=textos, fill_color=fondos, line_color=SURFACE, align="left", height=48,
               font=dict(size=11, color=TEXT)),
))
fig_desvio.update_layout(
    title=dict(text=f"Los 10 desvíos más grandes de cada año · con al menos {MIN_VOTOS_UTILES} votos útiles",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT), paper_bgcolor=PAGE,
    width=1300, height=640, margin=dict(l=20, r=20, t=60, b=20),
)
fig_desvio.show(config={"displaylogo": False})

**Cómo leer estos desvíos.** Casi todos los de la tabla tienen `|idp| > 1`: están
en los **bordes** del eje, no en el medio. Es un efecto conocido del modelo: un
diputado queda ubicado por las votaciones que lo separan de sus vecinos, y los
que están en un extremo no tienen a nadie más allá. Si todo el ala dura del
oficialismo (o de la oposición) vota siempre igual, el modelo sabe que están
"muy a la derecha" (o a la izquierda), pero no puede distinguir si uno está en
1.2 o en 1.6, porque no hay ninguna votación que los parta. El desvío alto no
significa que voten de forma errática; significa lo contrario: votan tan en
bloque que no hay datos para ordenarlos entre sí.

Leído así, la tabla cuenta quién fue el polo más compacto cada año:

- **2022 y 2023:** todo el top 10 es del Frente de Todos, en el extremo
  izquierdo. El ala dura del oficialismo votaba como un solo bloque.
- **2024:** el top se reparte entre Unión por la Patria y La Libertad Avanza,
  los dos polos del primer año de Milei. Los desvíos son más chicos (≈ 0.3–0.4)
  porque fue el año con más votaciones (más de 150 votos útiles por diputado):
  con más datos, todas las estimaciones se afinan.
- **2025:** se mezclan dos causas. Hay extremos de UxP y LLA con muchos votos
  (Valdés, Allende, Marín), y también diputados que asumieron en diciembre y
  tienen solo 17 o 18 votos útiles (Goitia, Grabois, Cámpora, Gruber): menos
  votos, más incertidumbre.
- **2026:** casi todo el top es de La Libertad Avanza, en el extremo derecho:
  ahora el bloque más disciplinado es el oficialismo.

Los desvíos no se comparan bien entre años: dependen de cuántas votaciones
hubo (en 2023 la mediana es ≈ 0.4 con unos 30 votos útiles por diputado; en
2024, ≈ 0.14 con más de 150).

## Dot plot (Wilkinson): distribución por bloque



In [ ]:
ANCHO_BIN = 0.06

dd = df.dropna(subset=["bloque"])

In [ ]:
def apilar(valores, ancho_bin):
    """Wilkinson: redondea cada valor al bin mas cercano y devuelve, para cada
    punto, en que nivel de la pila de ese bin le toca (0, 1, 2, ...)."""
    binned = (valores / ancho_bin).round() * ancho_bin
    nivel = binned.groupby(binned).cumcount()
    return binned, nivel


def mayor_hueco(g):
    gaps = g["idp_bayes"].diff()
    i_max = gaps.idxmax()
    antes, despues = g.loc[i_max - 1], g.loc[i_max]
    return gaps.loc[i_max], (antes["idp_bayes"] + despues["idp_bayes"]) / 2, antes["nombre"], despues["nombre"]


dots, huecos = {}, {}
for anio, g in dd.groupby("anio"):
    g = g.sort_values("idp_bayes").reset_index(drop=True)
    g["x_bin"], g["nivel"] = apilar(g["idp_bayes"], ANCHO_BIN)
    dots[anio], huecos[anio] = g, mayor_hueco(g)
    hueco, _, izq, der = huecos[anio]
    print(f"{anio}: mayor hueco {hueco:.2f}, entre {izq} y {der}")

In [ ]:
fig2 = go.Figure()

trazas_por_anio = {}
for anio, g in dots.items():
    trazas_por_anio[anio] = []
    for bloque_val, gb in g.groupby("bloque_mostrado"):
        trazas_por_anio[anio].append(len(fig2.data))
        fig2.add_trace(go.Scatter(
            x=gb["x_bin"], y=gb["nivel"],
            mode="markers",
            marker=dict(size=9, color=COLOR_BLOQUE[bloque_val], line=dict(width=0)),
            name=f"{bloque_val} ({len(gb)})",
            legendrank=1000 if bloque_val == "Otros bloques" else gb["idp_bayes"].median() * 100 + 500,
            customdata=list(zip(gb["nombre"], gb["bloque"], gb["idp_bayes"])),
            hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>idp = %{customdata[2]:.2f}<extra></extra>",
            hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
        ))


def vista_anio(anio):
    hueco, x_hueco, _, _ = huecos[anio]
    visibles = [i in trazas_por_anio[anio] for i in range(len(fig2.data))]
    layout = dict(
        shapes=[dict(type="line", x0=x_hueco, x1=x_hueco, yref="paper", y0=0, y1=1,
                     line=dict(color=MUTED, width=1, dash="dot"))],
        annotations=[dict(x=x_hueco, y=0.85, yref="paper", text=f"mayor hueco: {hueco:.2f}", showarrow=False,
                          font=dict(size=11, color=TEXT2), xanchor="left", xshift=6)],
    )
    return visibles, layout


inicial = anios[-1]
visibles, layout_inicial = vista_anio(inicial)
for traza, visible in zip(fig2.data, visibles):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=[{"visible": vista_anio(a)[0]}, vista_anio(a)[1]])
           for a in anios]

x_max = math.ceil(dd["idp_bayes"].abs().max() * 10) / 10 + ANCHO_BIN
fig2.update_xaxes(range=[-x_max, x_max], zeroline=True, zerolinecolor=BASELINE, zerolinewidth=1,
                   gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig2.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)

fig2.update_layout(
    **layout_inicial,
    title=dict(text="Puntos ideales (Bayes) por bloque \u00b7 C\u00e1mara de Diputados, 2022\u20132026",
               x=0.01, y=0.97, yanchor="top", font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=520, width=1000,
    margin=dict(l=40, r=20, t=110, b=40),
    legend=dict(orientation="h", y=-0.12, x=0),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.04, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig2.show(config={"displaylogo": False, "scrollZoom": False})

**Qué significa el hueco de 0.71 entre Giordano y Grabois (2026).** El hueco no
separa a dos personas sino a dos grupos: a la izquierda quedan solo los cinco
diputados del Frente de Izquierda (Del Plá, Bregman, Pitrola, Del Caño y
Giordano, el más "moderado" de ellos), y a la derecha los otros 252, empezando
por Grabois, Máximo Kirchner y Taiana, que encabezan Unión por la Patria. Pero
no es la brecha política más grande de la Cámara: Giordano y Grabois votan igual
en el 95% de las votaciones que comparten. De las 107 votaciones de 2026, el
FIT se separa de UxP en solo 6, y son todas del mismo tipo: tratados y acuerdos
internacionales (Mercosur–Unión Europea, Mercosur–Singapur, extradición con
Chile, traslado de condenados con Italia, cooperación jurídica, convenio con
Francia), que el FIT vota en contra prácticamente solo, frente a toda la Cámara.

Eso alcanza para el modelo: en cada una de esas votaciones el corte cae entre el
FIT y todos los demás, y como del otro lado del FIT no hay nadie que lo
"sostenga", el modelo lo empuja hacia afuera (el mismo efecto de borde que
explica los desvíos grandes en la tabla de posiciones de más arriba). El hueco
mide, entonces, una diferencia **consistente pero acotada**: el FIT se distingue
de UxP en un tema puntual (el rechazo a los acuerdos de libre comercio y a los
tratados con otros gobiernos), no en todo. La división principal de la Cámara,
entre UxP (mediana −0.84) y LLA (mediana +0.99), es bastante más grande, pero no
aparece como hueco porque el espacio entre los dos está ocupado por Provincias
Unidas, el PRO, la UCR y los bloques provinciales. Lo mismo pasa en otros años:
el hueco más grande suele aislar a un grupo chico de un extremo (en 2024, cinco
diputados a la izquierda de todos), más que partir la Cámara en dos.

PENDIENTE - IDEA: Así como tenemos el espectro de diputados... Podríamos hacer un espectro de proyectos. Desde el 2022 al 2026, hacer un dotplot similar al de arriba pero ubicando los proyectos de acuerdo a que tan "de derecha" son o que tan "de izquierda" son. Esto se saca por la adhesión que tuvieron en la derecha/izquierda y rechazo que tuvieron en derecha/izquierda.

PENDIENTE - IDEA: Proyectos que no respetaron la polaridad derecha-izquierda. Es decir, proyectos que fueron muy divisivos pero tiene a una cantidad relevante de gente de izquierda votando igual que derecha. En clase nombraron la ley de aborto, por ejemplo, como un proyecto que tiene ejes o dimensiones que escapan al espectro derecha izquierda. A mi se me ocurre ley de presupuesto general (puede haber gente de izquierda a favor para evitar la discresionalidad del poder Ejecutivo VS gente de izquierda en contra porque avala el ajuste) 

## Mapa de proyectos: corte, discriminación y dirección

El modelo de Clinton, Jackman y Rivers no solo ubica legisladores: cada votación
*j* también tiene sus parámetros, en `P(Sí) = Φ(β_j · θ_i − α_j)`:

- **Corte** `α_j / β_j`: el punto del eje izquierda–derecha donde un legislador
  queda indiferente entre Sí y No. Cerca de 0 es una votación que parte la Cámara
  al medio; lejos de 0 es una votación donde solo un extremo quedó del otro lado
  (qué tan *extrema* es).
- **Discriminación** `|β_j|`: qué tan nítido es ese corte. Alta = el voto se
  explica casi del todo por la ideología (izquierda de un lado, derecha del
  otro); cerca de 0 = el eje izquierda–derecha no explica el voto (otros clivajes:
  territoriales, de conciencia, etc.).
- **Dirección** (color): el `idp_bayes` promedio de los que votaron **Sí**. Rojo =
  la apoyó la izquierda; azul = la apoyó la derecha. Es otra forma de leer el
  signo de β, en unidades de punto ideal.

`ideal_pscl.py` muestrea β y α pero no los guarda, así que acá se reestiman sin
volver a correr la cadena: para cada votación, un probit de los votos (Sí = 1,
No = 0, el resto afuera) contra los `idp_bayes` ya estimados, con el mismo prior
N(0, 25) del modelo (estimación MAP). El prior es también el que pone techo a β
en las votaciones con separación perfecta (≈ 11). Las votaciones unánimes quedan
afuera, igual que en `preparar_pscl()`.

In [ ]:
info = pd.read_csv("datos/info_sesiones_dt.csv")

votos = (ses.assign(y=ses["voto"].map({"AFIRMATIVO": 1.0, "NEGATIVO": 0.0}))
            .dropna(subset=["y"])
            .merge(df[["nombre", "anio", "idp_bayes"]], left_on=["legislador", "yr"], right_on=["nombre", "anio"]))


def parametros_votacion(g):
    x, y = g["idp_bayes"].to_numpy(), g["y"].to_numpy()
    if y.min() == y.max():
        return None
    s = 2 * y - 1
    menos_log_post = lambda p: -log_ndtr(s * (p[0] * x - p[1])).sum() + 0.02 * (p ** 2).sum()
    beta, alpha = minimize(menos_log_post, [0.0, 0.0], method="BFGS").x
    return pd.Series({"beta": beta, "alpha": alpha, "corte": alpha / beta,
                      "discriminacion": abs(beta), "idp_si": x[y == 1].mean(),
                      "afirmativos": int(y.sum()), "negativos": int(len(y) - y.sum())})


proyectos = (votos.groupby(["anio", "id_votacion"]).apply(parametros_votacion).dropna().reset_index()
                  .merge(info[["id_votacion", "titulo", "fecha", "resultado"]], on="id_votacion"))
proyectos.describe().round(2)

In [ ]:
RANGO_CORTE = 2.5

visibles = proyectos[proyectos["corte"].abs() <= RANGO_CORTE]
print(f"{len(proyectos) - len(visibles)} de {len(proyectos)} votaciones con |corte| > {RANGO_CORTE} quedan fuera "
      f"del gráfico (casi unánimes, con discriminación baja: el corte no está bien definido)")

fig3 = go.Figure()
for anio in anios:
    g = visibles[visibles["anio"] == anio]
    fig3.add_trace(go.Scatter(
        x=g["corte"], y=g["discriminacion"], mode="markers", name=str(anio),
        marker=dict(size=9, color=g["idp_si"], colorscale=[[0, IZQUIERDA], [0.5, GRID], [1, DERECHA]],
                    cmin=-1, cmax=1, cmid=0, line=dict(width=1, color=SURFACE),
                    colorbar=dict(title=dict(text="idp medio<br>de los Sí", font=dict(size=11, color=TEXT2)),
                                  tickvals=[-1, 0, 1], ticktext=["izquierda", "centro", "derecha"],
                                  tickfont=dict(size=10, color=MUTED), thickness=12, len=0.6)),
        customdata=list(zip(g["titulo"], g["fecha"], g["resultado"], g["idp_si"], g["afirmativos"], g["negativos"])),
        hovertemplate=("<b>%{customdata[0]}</b><br>%{customdata[1]} · %{customdata[2]} "
                       "(%{customdata[4]:.0f} Sí / %{customdata[5]:.0f} No)<br>"
                       "corte = %{x:.2f} · discriminación = %{y:.2f} · idp medio Sí = %{customdata[3]:.2f}<extra></extra>"),
        hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
    ))

botones = [dict(label="Todos", method="update", args=[{"visible": [True] * len(anios)}])]
botones += [dict(label=str(a), method="update", args=[{"visible": [b == a for b in anios]}]) for a in anios]

fig3.update_xaxes(title=dict(text="corte (α/β) · ← izquierda · derecha →", font=dict(size=12, color=TEXT2)),
                  range=[-RANGO_CORTE, RANGO_CORTE], zeroline=True, zerolinecolor=BASELINE,
                  gridcolor=GRID, tickfont=dict(size=11, color=MUTED))
fig3.update_yaxes(title=dict(text="discriminación |β|", font=dict(size=12, color=TEXT2)),
                  rangemode="tozero", gridcolor=GRID, zeroline=False, tickfont=dict(size=11, color=MUTED))
fig3.update_layout(
    title=dict(text="Votaciones 2022–2026: dónde cortan, qué tan nítido y quién las apoyó",
               x=0.01, font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=600, width=1000, showlegend=False,
    margin=dict(l=60, r=20, t=110, b=60),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, x=0.01, xanchor="left",
                      y=1.02, yanchor="bottom", showactive=True, pad=dict(r=4, t=0),
                      bgcolor=SURFACE, bordercolor=GRID, font=dict(size=11, color=TEXT2))],
)
fig3.show(config={"displaylogo": False, "scrollZoom": False})




**Cómo leer un punto, paso a paso** (cada punto es una votación, no un diputado):

1. **Altura**: ¿la votación fue ideológica? Arriba, sí (el voto se predice con el
   punto ideal); abajo, no.
2. **Posición horizontal**: ¿partió la Cámara al medio (centro) o dejó del otro
   lado solo a un extremo (costados)?
3. **Color**: ¿quién la impulsó? Rojo, la izquierda; azul, la derecha; gris, casi
   todos.

Cuatro ejemplos, uno por zona:

- **Arriba al centro: grieta pura.** Los artículos de la Ley Bases (2024): corte
  ≈ 0.09, discriminación ≈ 11.7, 134 a 119. La Cámara partida justo al medio y
  casi nadie votó distinto de lo que predecía su posición. Azul: la apoyó la
  derecha (idp medio de los Sí = +0.82).
- **Arriba a la izquierda: solo la punta izquierda votó distinto.** Convenio con
  Austria para eliminar la doble imposición (2024): 224 a 5, corte −1.8. Los 5 No
  son el Frente de Izquierda; todo el resto votó Sí, por eso el color es gris.
- **Arriba a la derecha: solo la punta derecha votó distinto.** Ley de Respuesta
  Integral al VIH, Hepatitis Virales, ITS y TBC (2022): 241 a 8, corte +1.53. Los
  8 No están en el extremo derecho.
- **Abajo: el eje izquierda–derecha no explica el voto.** Emergencia en
  discapacidad, nuevo artículo 2 (2025): 123 a 82, pero con discriminación de
  0.21: hubo Sí y No en todo el espectro. Pesaron otras cosas (provincias,
  negociaciones puntuales), no la ideología.

## Grafos de cercanía y comunidades

Lo mismo que `04_redes.py` (el script de redes de la materia), pero dibujado acá
con plotly, con una solapa por año. Se parte de `datos/distancias_idps.csv.gz`: el producto
cartesiano de diputados de cada año, con la distancia entre sus puntos ideales
(`|idp_bayes A - idp_bayes B|`).

Para cada año:

1. Se arma la red completa: un nodo por diputado y una arista por par, con la
   distancia como peso. Sobre esa red se calcula la **intermediación**
   (betweenness) de cada diputado.
2. Se **poda**: se borran las aristas con distancia mayor a `PUNTO_CORTE` (0.4),
   y después los diputados que quedan con menos de 3 uniones.
3. Se detectan **comunidades** con fast greedy (peso `1 / distancia`, así una
   unión corta pesa más), cortando donde la modularidad es máxima.

En el dibujo, cada comunidad (las 8 más grandes; el resto en gris) toma el
color de su bloque mayoritario, el mismo que en el resto del trabajo; el
tamaño del nodo es su intermediación y se etiquetan los 5 diputados con más
intermediación (pasar el mouse sobre cualquier nodo muestra nombre, bloque e
intermediación). La leyenda dice, para cada comunidad, qué bloque es mayoría y en
qué proporción. La posición sale de Fruchterman-Reingold, así que la forma
cambia un poco entre corridas (la estructura no).

In [ ]:
PUNTO_CORTE = 0.4
MAX_COMUNIDADES = 8
OTRAS, ARISTA = "#c3c2b7", "#898781"

distancias = pd.read_csv("datos/distancias_idps.csv.gz").rename(
    columns={"nombre1": "Source", "nombre2": "Target", "distancia": "weight"})
distancias = distancias[distancias.Source < distancias.Target]


def armar_grafo(anio):
    dist = distancias[distancias.anio == anio]
    dips = bloques[bloques.yr == anio]
    g = ig.Graph.DataFrame(dist[["Source", "Target", "weight"]], directed=False,
                           vertices=dips[["legislador"]], use_vids=False)
    g.vs["bloque"] = list(dips.bloque)
    return g


def podar(g):
    g = g.copy()
    g.simplify(combine_edges="sum")
    g.delete_edges(g.es.select(weight_gt=PUNTO_CORTE))
    g.simplify(combine_edges="sum")
    g.delete_vertices(g.vs.select(_degree_lt=3))
    return g


def intermediacion(g):
    n = g.vcount()
    return [b * 2 / ((n - 1) * (n - 2)) for b in g.betweenness(directed=False, weights="weight")]


def comunidades(g):
    arbol = g.community_fastgreedy(weights=[1 / w for w in g.es["weight"]])
    membresia = ig.community_to_membership(arbol.merges, g.vcount(), g.vcount() - arbol.optimal_count)
    return ig.VertexClustering(g, membresia)

In [ ]:
def calcular_red(anio):
    g = armar_grafo(anio)
    g.vs["betweenness"] = intermediacion(g)
    g = podar(g)
    x = comunidades(g)
    por_tamanio = sorted(range(len(x)), key=lambda c: -len(x[c]))
    xs, ys = zip(*g.layout_fruchterman_reingold(weights=[1 / w for w in g.es["weight"]], niter=2000).coords)
    bloque = ["Frente de Izquierda" if "izquierda" in b.lower() else NOMBRE_BLOQUE.get(b, b) for b in g.vs["bloque"]]
    nodos = pd.DataFrame({"nombre": [n.replace('"', "").title() for n in g.vs["name"]], "bloque": bloque,
                          "betweenness": g.vs["betweenness"], "x": xs, "y": ys,
                          "comunidad": [por_tamanio.index(m) for m in x.membership]})
    aristas = [e.tuple for e in g.es]
    return nodos, aristas, len(x)


redes = {anio: calcular_red(anio) for anio in anios}

En 2025 hay más diputados que bancas (257) porque el 10 de diciembre cambió la
composición de la Cámara: en `legisladores_bloque_provincia.csv` aparecen tanto
los que salieron como los que entraron ese año.

In [ ]:
fig4 = go.Figure()
trazas_red = {}
titulos_red = {}
etiquetas_red = {}

for anio, (nodos, aristas, n_comunidades) in redes.items():
    trazas_red[anio] = []
    titulos_red[anio] = (f"Grafo de cercan\u00eda \u00b7 {anio} \u00b7 {len(nodos)} diputados, "
                         f"{len(aristas)} uniones, {n_comunidades} comunidades")

    ex, ey = [], []
    for a, b in aristas:
        ex += [nodos.x[a], nodos.x[b], None]
        ey += [nodos.y[a], nodos.y[b], None]
    trazas_red[anio].append(len(fig4.data))
    fig4.add_trace(go.Scatter(x=ex, y=ey, mode="lines", line=dict(color="rgba(137, 135, 129, 0.06)", width=0.5),
                                hoverinfo="skip", showlegend=False))

    nodos = nodos.assign(grupo=nodos["comunidad"].clip(upper=MAX_COMUNIDADES))
    for grupo, gn in nodos.groupby("grupo"):
        if grupo < MAX_COMUNIDADES:
            bloque = gn["bloque"].mode()[0]
            color = COLOR_BLOQUE.get(bloque, OTRAS)
            nombre = f"{len(gn)} dip. \u00b7 {(gn['bloque'] == bloque).mean():.0%} {bloque}"
        else:
            color = OTRAS
            nombre = f"{len(gn)} dip. \u00b7 otras {n_comunidades - MAX_COMUNIDADES} comunidades"
        trazas_red[anio].append(len(fig4.data))
        fig4.add_trace(go.Scatter(
            x=gn["x"], y=gn["y"], mode="markers", name=nombre,
            marker=dict(size=(10 + 500 * gn["betweenness"]) ** 0.5 * 1.3, color=color,
                        line=dict(width=0.6, color=SURFACE)),
            customdata=list(zip(gn["nombre"], gn["bloque"], gn["betweenness"])),
            hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>intermediaci\u00f3n = %{customdata[2]:.3f}<extra></extra>",
            hoverlabel=dict(bgcolor=TEXT, font_color=SURFACE, font_size=12),
        ))

    top = nodos.nlargest(5, "betweenness").sort_values("x")
    etiquetas_red[anio] = [dict(x=n.x, y=n.y, text=n.nombre, showarrow=True, arrowhead=0, arrowwidth=0.6,
                                arrowcolor=MUTED, ax=0, ay=-40 - 28 * k, font=dict(size=11, color=TEXT),
                                bgcolor=SURFACE, borderpad=1)
                           for k, n in enumerate(top.itertuples())]


def vista_red(anio):
    visibles = [i in trazas_red[anio] for i in range(len(fig4.data))]
    return [{"visible": visibles}, {"title.text": titulos_red[anio], "annotations": etiquetas_red[anio]}]


inicial = anios[-1]
for traza, visible in zip(fig4.data, vista_red(inicial)[0]["visible"]):
    traza.visible = visible

botones = [dict(label=str(a), method="update", args=vista_red(a)) for a in anios]

fig4.update_xaxes(visible=False)
fig4.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
fig4.update_layout(
    annotations=etiquetas_red[inicial],
    title=dict(text=titulos_red[inicial], x=0.01, y=0.97, yanchor="top",
               font=dict(size=18, color=TEXT, family=FONT)),
    font=dict(family=FONT, color=TEXT),
    paper_bgcolor=PAGE, plot_bgcolor=SURFACE,
    height=750, width=1100,
    margin=dict(l=20, r=20, t=110, b=20),
    legend=dict(itemsizing="constant", title=dict(text="Comunidad \u00b7 bloque mayoritario", font=dict(size=11, color=TEXT2)),
                x=1.0, y=1.0, xanchor="left", yanchor="top", font=dict(size=11, color=TEXT2)),
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=anios.index(inicial),
                      x=0.01, xanchor="left", y=1.04, yanchor="bottom", showactive=True,
                      pad=dict(r=4, t=0), bgcolor=SURFACE, bordercolor=GRID,
                      font=dict(size=11, color=TEXT2))],
)

fig4.show(config={"displaylogo": False, "scrollZoom": False})

## Nube de palabras: qué proyectos apoya cada lado

Con los títulos de las votaciones del mapa de proyectos, dos nubes: las palabras
de los proyectos que apoyó la izquierda y las de los que apoyó la derecha. El
lado sale del mismo `idp_si` del mapa (el `idp_bayes` promedio de los que
votaron Sí): izquierda si es ≤ −0.4, derecha si es ≥ 0.4; lo del medio
(tratados, convenios, votaciones con apoyo amplio) queda afuera.

Antes de contar:

- Se descartan las votaciones de trámite (apartamientos de reglamento,
  mociones, habilitaciones, reconsideraciones, plan de labor, insistencias):
  sus títulos tienen nombres de diputados, no temas.
- Cada proyecto cuenta una sola vez: el Presupuesto o la Ley Bases se votaron
  decenas de veces (en general y por capítulo) y si no taparían todo lo demás.
  Las votaciones se agrupan por número de orden del día y año, y de cada
  proyecto se toman las palabras de todos sus títulos.
- Se sacan las palabras de procedimiento ("dictamen de mayoría", "votación en
  general", "modificación", "creación", números romanos de capítulos,
  abreviaturas) y las tildes al comparar (los títulos mezclan CREACION y
  CREACIÓN).

El tamaño de cada palabra es en cuántos proyectos de ese lado aparece. Hace
falta el paquete `wordcloud` (`pip install wordcloud`).

In [ ]:
UMBRAL_IDP = 0.4
TRAMITE = r"^(APARTAMIENTO|MOCI[ÓO]N|HABILITACI[ÓO]N|RECONSIDERACI[ÓO]N|PLAN DE LABOR|VOTACI[ÓO]N DESARROLLO|INSISTENCIA)"
NO_TEMA = set("""
ley leyes vot votacion gral general part particular dict dictamen may mayoria minoria modif modificacion modificaciones
mod creacion aprobacion establecimiento declaracion insistir insistencia proyecto sancion honorable camara diputados
senado hsn hcdn acep aceptacion introd introducidas nacional nacionales nac rep republica argentina argentinos art
articulo arts titulo capitulo inciso expte exptes exp del las los por para con una uno sus que entre sobre como desde
ante sin hasta otros otras diversas diversos distintos todo toda total totalidad dur susp corresp ambito marco relativas
relacionadas cuestiones redaccion texto propuesto venido revision materia nuevo nueva pens cen lib reg cap ant proy arg
inst prov desarr adm cod simpl presup imp decl transp fed preserv amb reinc reit unif derog acreed tecn
""".split())


def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


tematicas = proyectos[~proyectos["titulo"].str.upper().str.match(TRAMITE)]
od = tematicas["titulo"].str.upper().str.extract(r"^O\.?\s*D\.?\s*(\d+)")[0]
tematicas = tematicas.assign(proyecto=tematicas["anio"].astype(str) + "-" + od.fillna(tematicas["titulo"]))
por_proyecto = tematicas.groupby("proyecto").agg(titulos=("titulo", " ".join), idp_si=("idp_si", "mean"))
por_proyecto["lado"] = pd.cut(por_proyecto["idp_si"], [-np.inf, -UMBRAL_IDP, UMBRAL_IDP, np.inf],
                              labels=["izquierda", "centro", "derecha"])

formas = collections.defaultdict(collections.Counter)
frecuencias = {"izquierda": collections.Counter(), "derecha": collections.Counter()}
for titulos, lado in zip(por_proyecto["titulos"], por_proyecto["lado"]):
    if lado == "centro":
        continue
    palabras = set()
    for palabra in re.findall(r"[^\W\d_]{3,}", titulos.lower()):
        clave = sin_tildes(palabra)
        if clave not in NO_TEMA and not re.fullmatch(r"[ivxlc]+", clave):
            formas[clave][palabra] += 1
            palabras.add(clave)
    frecuencias[lado].update(palabras)

por_proyecto["lado"].value_counts()

In [ ]:
fig5, ejes = plt.subplots(1, 2, figsize=(14, 6), facecolor=PAGE)
for eje, (lado, color) in zip(ejes, [("izquierda", IZQUIERDA), ("derecha", DERECHA)]):
    pesos = {formas[clave].most_common(1)[0][0]: n for clave, n in frecuencias[lado].items()}
    nube = WordCloud(width=900, height=700, background_color=SURFACE, color_func=lambda *a, **k: color,
                     prefer_horizontal=1.0, random_state=42).generate_from_frequencies(pesos)
    eje.imshow(nube, interpolation="bilinear")
    eje.set_axis_off()
    n = (por_proyecto["lado"] == lado).sum()
    eje.set_title(f"Apoyados por la {lado} ({n} proyectos)", loc="left", fontsize=14, color=TEXT)
fig5.suptitle(f"Palabras en los títulos de los proyectos según quién los votó (|idp medio de los Sí| ≥ {UMBRAL_IDP})",
              x=0.01, ha="left", fontsize=16, color=TEXT)
plt.tight_layout()
plt.show()